# Demo: Fine-tuning BERT, GPT-2, and T5

## 1. BERT for sentiment classification

- Why BERT?: strong at classification with minimal training
- Task: sentence-level sentiment classification
- Dataset (Hugging Face datasets names): SST-2 (GLUE) – glue

## 2. GPT-2 for dialogue/response generation

- Why GPT?: natural for next-token generation
- Task: short-turn response generation
- Dataset: DailyDialog – daily_dialog

## 3. T5 for dialogue summarization

- Why T5: designed for seq2seq; single text-to-text interface
- Task: summarization (text-to-text)
- Dataset: SAMSum – samsum

In [1]:
import os, json, random

import torch
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
    DataCollatorWithPadding
)

In [2]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# Pre-download SST-2 dataset
raw_data = load_dataset('glue', 'sst2')
print('SST-2 done downloading.')

SST-2 done downloading.


In [4]:
model_name = 'prajjwal1/bert-tiny'
output_dir = 'outputs/bert-tiny-sst2'

In [5]:
# Tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
collator = DataCollatorWithPadding(tokenizer)

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
# Tokenize and preprocess data
def preprocess(examples):
    return tokenizer(examples['sentence'], truncation=True, max_length=128)

tokenized_data = raw_data.map(preprocess, batched=True)

In [ ]:
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1': float(f1_score(labels, preds))
    }

In [ ]:
# Train
args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to=[]
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['validation'],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
os.makedirs(output_dir, exist_ok=True)
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
metrics = trainer.evaluate(tokenized_data['validation'])
metrics

In [ ]:
with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved to', output_dir, '', metrics)

In [ ]:
# Quick classification
samples = [
    "I absolutely loved this movie!",
    "This was incredibly boring and a waste of time.",
    "Not bad, but could be better."
]

In [ ]:
model.eval()

In [ ]:
for t in samples:
    enc = tokenizer(t, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.nn.functional.softmax(logits, dim=-1).squeeze().tolist()
    pred = int(torch.argmax(logits, dim=-1).item())
    print(f"{t} -> {'positive' if pred==1 else 'negative'} (p={probs[pred]:.3f}) | probs={probs}")